# 03 — What does Semantica add?

Think of Semantica ContextGraph as a **decision card box with an index**.

Notebook 2 made one run readable. Notebook 3 asks what we can do after many runs have
been stored as native Semantica Decisions. We keep only four useful questions:

1. What is on one decision card?
2. Can we find similar cards?
3. Did the same question receive different answers?
4. If a policy changes, which old cards must a human re-check?

A card never replaces the raw trace. Its provenance pointer is the receipt number that
lets us go back to the original run.


**Checked reading copy.** The saved outputs below come from completed real-provider runs. Implementation cells are omitted here for readability; open [`03_semantica_decision_intelligence.ipynb`](./03_semantica_decision_intelligence.ipynb) to inspect or rerun the code.


**Loaded:** 7 runs → 62 decision cards.

## 1. One decision card

Semantica's native unit is simple: `category`, `scenario`, `reasoning`, `outcome`, and
`confidence`. The main fields are human-readable and de-identified. Exact dates, note
locators, and field-level provenance remain behind the card in the audit record.


| Decision field | Recorded value |
|---|---|
| Category | standing |
| Scenario | This pathology document is dated [date] and reports atypical cells suspicious for squamous cell carcinoma with biopsy correlation recommended; its standing for the diagnosis date is unresolved. |
| Reasoning | The document uses only an ambiguous suspicious term, so it does not itself establish the diagnosis under the evidence rule; a later positive biopsy and separate physician impression must be compared. Basis used: chart evidence and the supplied task contract. |
| Outcome | MERELY_MENTIONS |
| Confidence | 1.0 |

**How to read confidence:** it measures reconstruction stability across passes, not clinical correctness. The card is still something a human must judge.

This is the chart-review equivalent of Semantica's vendor-selection example:

```python
graph.record_decision(
    category="standing",
    scenario="Can suspicious cytology alone establish the diagnosis date?",
    reasoning="The report is ambiguous and recommends confirmatory biopsy.",
    outcome="MERELY_MENTIONS",
    confidence=0.88,  # reconstruction stability, not correctness
)
```

## 2. Find similar decisions

Similarity is useful because a reviewer can inspect a small set of prior judgments
instead of searching every trace. It proposes comparison candidates; it does not say
that either judgment is correct.


**Query card:** This pathology document is dated [date] and reports atypical cells suspicious for squamous cell carcinoma with biopsy correlation recommended; its standing for the diagnosis date is unresolved.

| Run context | Comparable question | Outcome | Similarity |
|---|---|---|---|
| SYN0001 · Luna · Task only | For the requested initial physician diagnosis date, this earlier pathology document must be classified as establishing, merely mentioning, or neither. | MERELY_MENTIONS | 0.47 |
| SYN0001 · Terra · Task only | This pathology document was read as a candidate for diagnosis timing; the question is whether its result confirms a tumour diagnosis. | MERELY_MENTIONS | 0.46 |
| SYNX03 · Terra · Task + policy | This [date] fine-needle aspirate is a candidate earlier diagnosis witness; its suspicious wording may be ambiguous cytology and its standing depends on later c… | CAN_ESTABLISH | 0.46 |

**So what?** These are good cards to compare for consistency. A similarity score closer to 1 means more alike; it is not an accuracy score.

## 3. Find the same question with different answers

“Similar” is broad. A stronger audit signal is **the same case, the same evidence note,
and the same atomic question**, but a different outcome.

The stored runs contain exactly that pattern for one `SYN0001` oncology note. In the
task-only arm, no clinical policy was provided, so the models had to use their own
judgment.


| Case | Model | Answer to same question | Recorded reasoning | Policy grounding |
|---|---|---|---|---|
| SYN0001 | Terra | MERELY_MENTIONS | The assessment calls the lung mass clinically malignant but explicitly says tissue sampling will be performed for confirmation, so it documents suspicion rather than an established adenocarcinoma diagnosis. Basis used: chart evidence and the model's own judgment. | None — task only |
| SYN0001 | Luna | CAN_ESTABLISH | The physician's [date] service date is explicit, and the assessment states the right-upper-lobe mass clinically represents malignancy; this is the earliest explicit physician diagnosis found. Basis used: chart evidence and the supplied task contract. | None — task only |

**So what?** This is a guideline-gap candidate: the same evidence received different standings when the task supplied no rule. A human should decide the desired rule, then add it to the guideline.

## 4. If a policy changes, what must be re-checked?

Suppose we tighten this policy:

> If a physician diagnosis predates tissue confirmation, use the earlier physician
> date; later tissue confirmation does not reset an already established diagnosis.

Semantica stores versioned Policy nodes and direct `APPLIED_POLICY` links. That lets us
retrieve the historical decisions that cited this policy.


- **Historical runs to revisit:** 3
- **Directly bound decisions:** 16
- **Parts of the review involved:** Decide whether to stop, Judge evidence, Choose the final answer, Search / choose notes, Resolve conflicts
- **Meaning:** this is a re-audit queue, not a prediction that every answer changes.

## The whole point

| Without the ContextGraph | With the ContextGraph |
|---|---|
| Read one trace at a time | Retrieve a small set of comparable Decision cards |
| Notice only final-answer differences | Localize disagreement to one evidence judgment |
| Guess which old runs a rule touched | Ask Semantica for directly policy-bound decisions |
| Trust a summary | Follow the card's provenance back to the raw trace |

**Use the graph to route human attention. Use provenance and raw trace to decide what
really happened. Use a qualified reviewer to decide what is clinically correct.**
